In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("ProcessUserProfiles") \
    .master("local[*]") \
    .getOrCreate()

26/02/15 20:41:54 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
profiles_path = "data/user_profiles/user_profiles.json"

In [4]:
df_profiles = spark.read.json(profiles_path)

In [5]:
print("Profiles count:", df_profiles.count())

Profiles count: 47469


In [6]:
df_profiles.printSchema()

root
 |-- birth_date: string (nullable = true)
 |-- email: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- state: string (nullable = true)



In [7]:
df_profiles.show(5, truncate=False)

+----------+----------------------------+----------------+------------------+-----+
|birth_date|email                       |full_name       |phone_number      |state|
+----------+----------------------------+----------------+------------------+-----+
|1989-01-11|norma_fisher@example.com    |Norma Fisher    |(604)876-4759x3824|Ohio |
|1991-06-08|jorge_sullivan@example.com  |Jorge Sullivan  |(194)892-4115     |Ohio |
|1965-07-27|elizabeth_woods@example.com |Elizabeth Woods |815.659.3877x8408 |Idaho|
|1980-06-07|susan_wagner@example.com    |Susan Wagner    |1609753513        |Maine|
|1997-08-09|peter_montgomery@example.com|Peter Montgomery|001-332-871-1587  |Texas|
+----------+----------------------------+----------------+------------------+-----+
only showing top 5 rows


In [8]:
from pyspark.sql.functions import split, col


In [9]:
df_profiles = df_profiles.withColumn("first_name", split(col("full_name"), " ").getItem(0)) \
                         .withColumn("last_name", split(col("full_name"), " ").getItem(1))

In [10]:
df_profiles.select("first_name", "last_name", "birth_date", "state", "phone_number").show(5, truncate=False)

+----------+----------+----------+-----+------------------+
|first_name|last_name |birth_date|state|phone_number      |
+----------+----------+----------+-----+------------------+
|Norma     |Fisher    |1989-01-11|Ohio |(604)876-4759x3824|
|Jorge     |Sullivan  |1991-06-08|Ohio |(194)892-4115     |
|Elizabeth |Woods     |1965-07-27|Idaho|815.659.3877x8408 |
|Susan     |Wagner    |1980-06-07|Maine|1609753513        |
|Peter     |Montgomery|1997-08-09|Texas|001-332-871-1587  |
+----------+----------+----------+-----+------------------+
only showing top 5 rows


In [11]:
df_profiles.write.mode("overwrite").parquet("silver/user_profiles")

In [12]:
df_customers = spark.read.parquet("silver/customers")
df_profiles = spark.read.parquet("silver/user_profiles")

In [13]:
print("Customers count:", df_customers.count())

Customers count: 47469


In [14]:
print("Profiles count:", df_profiles.count())

Profiles count: 47469


In [15]:
df_customers.printSchema()

root
 |-- client_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- state: string (nullable = true)



In [16]:
df_profiles.printSchema()

root
 |-- birth_date: string (nullable = true)
 |-- email: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- state: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)



In [17]:
from pyspark.sql.functions import coalesce, col

In [19]:
df_enriched = df_customers.alias("c") \
    .join(
        df_profiles.alias("p"),
        on="email",
        how="left"
    ) \
    .select(
        col("c.client_id"),
        coalesce(col("c.first_name"), col("p.first_name")).alias("first_name"),
        coalesce(col("c.last_name"), col("p.last_name")).alias("last_name"),
        coalesce(col("c.state"), col("p.state")).alias("state"),
        col("c.email"),
        col("c.registration_date"),
        col("p.birth_date"),
        col("p.phone_number")
    )

df_enriched.show(10, truncate=False)

+---------+----------+----------+-----+----------------------------+-----------------+----------+---------------------+
|client_id|first_name|last_name |state|email                       |registration_date|birth_date|phone_number         |
+---------+----------+----------+-----+----------------------------+-----------------+----------+---------------------+
|26       |Sarah     |Villanueva|Idaho|sarah_villanueva@example.com|2022-08-03       |2000-11-28|551-590-0422x94568   |
|27       |Kimberly  |Myers     |Utah |kimberly_myers@example.com  |2022-08-01       |1969-08-31|(417)304-2814        |
|28       |Desiree   |Cain      |Maine|desiree_cain@example.com    |2022-08-05       |1986-11-21|(546)118-7755x1717   |
|31       |Whitney   |Stark     |Idaho|whitney_stark@example.com   |2022-08-05       |2004-05-20|(349)263-5110x873    |
|34       |Faith     |Cabrera   |Maine|faith_cabrera@example.com   |2022-08-04       |1995-04-11|+1-577-389-3055x50824|
|44       |Matthew   |Bell      |Texas|m

In [20]:
spark.stop()